In [13]:
import torch
import numpy as np
import gymnasium as gym
print("everything works! pytorch version", torch.__version__)

everything works! pytorch version 2.8.0


In [14]:
import pandas as pd
import glob

columns = [
    'time', 'missing_info', 'job_id', 'task_index', 'machine_id',
    'event_type', 'user', 'scheduling_class', 'priority',
    'cpu_request', 'memory_request', 'disk_request', 'different_machine'
]

files = sorted(glob.glob('trace_data/*.csv.gz'))
print(f"Reading {len(files)} files...")

frames = [pd.read_csv(f, header=None, names=columns) for f in files]
df = pd.concat(frames, ignore_index=True)
print(f"Total rows: {len(df):,}")

# clean submissions only
jobs = df[df['event_type'] == 0].dropna(
    subset=['cpu_request', 'memory_request']
).copy()

# time features
jobs['hours_elapsed'] = jobs['time'] / 1_000_000 / 3600

# TRIM to exactly 7 full days — removes the ragged edge
jobs = jobs[jobs['hours_elapsed'] < 7 * 24].copy()

jobs['hour_of_day'] = (jobs['hours_elapsed'] % 24).astype(int)
jobs['day_of_week'] = ((jobs['hours_elapsed'] // 24) % 7).astype(int)

print(f"Usable jobs (7 full days): {len(jobs):,}")
print(f"Time span: {jobs['hours_elapsed'].max():.1f} hours")

Reading 130 files...
Total rows: 36,851,860
Usable jobs (7 full days): 11,902,232
Time span: 168.0 hours


In [15]:
pivot = jobs.groupby(['day_of_week', 'hour_of_day']).size().unstack(fill_value=0)
print("Smallest bucket:", pivot.values.min(), "jobs")
print("Buckets with < 100 jobs:", (pivot.values < 100).sum(), "out of", pivot.size)

Smallest bucket: 11505 jobs
Buckets with < 100 jobs: 0 out of 168


In [16]:
import json
import numpy as np

# Level 2: separate stats for each (day, hour) combination
stats = {}

for day in range(7):
    stats[day] = {}
    for hour in range(24):
        bucket = jobs[(jobs['day_of_week'] == day) &
                      (jobs['hour_of_day'] == hour)]

        # arrival rate: jobs in this (day,hour) bucket
        # this is one week, so each bucket = 1 occurrence = jobs per hour
        arrival_rate = len(bucket)

        # scheduling class distribution (for deadlines)
        class_counts = bucket['scheduling_class'].value_counts(normalize=True)
        class_dist = [float(class_counts.get(c, 0.0)) for c in range(4)]

        stats[day][hour] = {
            'arrival_rate': float(arrival_rate),
            'avg_cpu': float(bucket['cpu_request'].mean()),
            'cpu_std': float(bucket['cpu_request'].std()),
            'avg_mem': float(bucket['memory_request'].mean()),
            'mem_std': float(bucket['memory_request'].std()),
            'class_distribution': class_dist,
        }

# bundle and save
trace_params = {
    'source': 'Google Cluster Trace 2011, 7 full days, part 0-129',
    'fidelity': 'per-day-per-hour (Level 2)',
    'stats': stats,
}

with open('trace_params.json', 'w') as f:
    json.dump(trace_params, f, indent=2)

print("Saved trace_params.json")

# show the weekly pattern — total jobs per day
print("\nTotal arrivals per day (shows weekly variation):")
for day in range(7):
    day_total = sum(stats[day][h]['arrival_rate'] for h in range(24))
    bar = '#' * int(day_total / 100000)
    print(f"  day {day}: {day_total:>10,.0f}  {bar}")

Saved trace_params.json

Total arrivals per day (shows weekly variation):
  day 0:  1,243,312  ############
  day 1:  4,871,034  ################################################
  day 2:  2,191,759  #####################
  day 3:  1,163,720  ###########
  day 4:    935,691  #########
  day 5:    817,787  ########
  day 6:    678,929  ######
